# Serve Qwen3.5-4B as an endpoint, and measure it

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/qwen3.5-serving/qwen3.5-serving-tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Take a model off the Hub, put it behind an OpenAI-compatible endpoint on a GPU you control, and check it the way you would check any service: with a scored request set and a load test, from code, before anyone points an app at it.

| Step | What happens |
|---|---|
| 1 | `serve.py` prefetches the weights into object storage once and deploys vLLM; the app scales to zero when idle |
| 2 | `sanity` asks twelve questions with known answers and scores them, with tokens and seconds on the same row |
| 3 | `benchmark` streams concurrent requests: tokens/s, time to first token, latency |
| 4 | `compare` runs thinking off vs on, so the cost of thinking is a number before it is a default |
| 5 | A Gradio chat UI that follows the endpoint wherever it is deployed |

This is leg 1 of the model-drop template (`tutorials/MODEL_DROP.md`). The README has the setup paths in detail.

---

## Setup

### Attach an accelerator first

vLLM needs an NVIDIA GPU: the Union cluster (`GPU=L40s`, the default) or `flyte start devbox --gpu` on an NVIDIA machine. Colab cannot host the app, and an Apple Silicon devbox cannot run vLLM; both can run the client steps (2 to 4) against a cluster endpoint with `LLM_BASE_URL` set below. To serve a model on the Mac's own GPU instead, see the `apple-silicon-serving` tutorial.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/qwen3.5-serving
    !uv pip install -r requirements.txt
    !uv pip install keyrings.alt
    !mkdir -p ~/.config/python_keyring && echo -e '[backend]\ndefault-keyring=keyrings.alt.file.PlaintextKeyring' > ~/.config/python_keyring/keyringrc.cfg
    %env TERM=dumb

### A cluster or devbox is required for step 1

vLLM is a deployment, so step 1 needs a Flyte or Union deployment with a GPU. Steps 2 to 4 are client tasks: they run on the cluster, or `--local` here against any endpoint (set `LLM_BASE_URL`).

In [ ]:
# Optional: connect to a Flyte or Union cluster, or a devbox on your machine.
# Skip this to use `--local` cells only.
#
# Don't have a cluster? Request demo access at https://union.ai/
#
# !flyte create config \
#     --endpoint <your-endpoint> \
#     --project flytesnacks \
#     --domain development \
#     --builder remote

In [ ]:
# If you are NOT deploying from this notebook, point the client tasks at an existing endpoint:
import os
# os.environ["LLM_BASE_URL"] = "https://<app-host>"

### Running the workflows

Every step is a plain `flyte run`, so the usual flags work:

| | |
|---|---|
| `flyte run --local ...` | Runs in this Python process. No cluster, no image build, no reports. |
| `flyte run ...` | Runs on the cluster or devbox you configured above: containers, the run graph and the live reports in the UI. |

Open the run URL that `flyte run` prints to see the report: that is where the charts are.

### 1. Serve

Prefetch once, deploy, scale to zero. Reuse a prefetch with `PREFETCH_RUN=<run name>`.

In [ ]:
!cat config.py

In [ ]:
!python serve.py

### 2. Is it right?

Twelve questions with gold answers, scored by a regex, with output tokens and seconds next to the score.

In [ ]:
!flyte run benchmark.py sanity
# no cluster: !flyte run --local benchmark.py sanity   (with LLM_BASE_URL set above)

### 3. How fast?

Concurrent streamed requests. The first one after idle includes the cold start.

In [ ]:
!flyte run benchmark.py benchmark --prompts 16 --concurrency 8

### 4. Thinking off vs on

Same endpoint, per-request switch. The report charts accuracy, tokens and seconds for both.

In [ ]:
!flyte run benchmark.py compare

### 5. A chat UI

In [ ]:
!python app_gradio.py
# no cluster: SERVER_URL=https://<app-host> python app_gradio.py --local

### 6. From any client

In [ ]:
import os
from openai import OpenAI

base = os.environ.get("LLM_BASE_URL", "https://<app-host>").rstrip("/") + "/v1"
client = OpenAI(base_url=base, api_key=os.environ.get("VLLM_API_KEY", "not-used"))
r = client.chat.completions.create(
    model="qwen3.5-4b",
    messages=[{"role": "user", "content": "What is a Flyte task, in one sentence?"}],
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(r.choices[0].message.content)

## What to take away

- The weights live in object storage and stream to the GPU; the image never contains the model, so a new model is a new prefetch.
- "Deployed" and "answers correctly" are different claims. `sanity` makes the second one cheap and puts cost on the same row.
- Thinking is a per-request switch with a measurable price. Decide with the `compare` report, not by default.